In [0]:
-- Gold Table Query: Join game_data, pitch_data, and strike_probability
CREATE MATERIALIZED VIEW IF NOT EXISTS mlb_tech_summit.mlb_gumbo_gold.game_report_mv
COMMENT 'Materialized View to power the Game Report Dashboard with pitch-level, game-level, and strike probability data'
AS
SELECT
    -- Game-level Metadata
    g.season,
    g.game_pk,
    g.official_date,
    g.home_team_name,
    g.away_team_name,
    g.home_team_record_wins AS home_team_wins,
    g.away_team_record_wins AS away_team_wins,
    g.home_team_record_losses AS home_team_losses,
    g.away_team_record_losses AS away_team_losses,
    g.venue_name,
    g.weather_condition,
    g.weather_temp,
    g.weather_wind,

    -- Pitch Event Metadata
    p.pitch_index,
    p.pitch_number,
    p.pitch_guid,
    p.inning,
    p.half_inning,
    p.is_pitch,
    p.call_description,                -- Ball, strike, foul, etc.
    p.position_x,                     -- Horizontal position at the plate
    p.position_z,                     -- Vertical position at the plate
    p.pitch_type_description,          -- Pitch type (e.g., fastball, slider)
    p.pitch_start_speed,               -- Pitch speed at release (mph)
    p.pitch_end_speed,                 -- Pitch speed at the plate (mph)
    p.pitch_spin_rate,                 -- Pitch spin rate (RPM)
    p.pitch_break_length,              -- Length of pitch break (inches)
    p.pitch_break_vertical,            -- Vertical break (inches)
    p.pitch_break_horizontal,          -- Horizontal break (inches)
    p.plate_time,                      -- Time for pitch to reach the plate (seconds)
    p.extension,                       -- Extension of pitcher's release (feet)
    
    -- Hit Data (if applicable)
    p.hit_launch_speed,                -- Exit velocity (mph)
    p.hit_launch_angle,                -- Launch angle (degrees)
    p.hit_total_distance,              -- Distance of the hit (feet)
    p.hit_trajectory,                  -- Hit trajectory type
    p.hit_hardness,                    -- Hardness rating of the hit
    
    -- Player Data
    p.batter_full_name,
    p.batter_side_description,         -- Batter's batting side
    p.pitcher_full_name,
    p.pitcher_hand_description,        -- Pitcher's throwing hand

    -- Strike Probability
    sp.called_strike_probability             -- Model-predicted probability of a strike

FROM
    mlb_tech_summit.mlb_gumbo_silver.game_data g
JOIN
    mlb_tech_summit.mlb_gumbo_silver.pitch_data p
ON
    g.game_pk = p.game_pk
LEFT JOIN
    mlb_tech_summit.mlb_gumbo_silver.strike_probability sp
ON
    g.game_pk = sp.game_pk
    AND p.at_bat_index = sp.at_bat_index
    AND p.pitch_index = sp.pitch_index
WHERE
    p.is_pitch = TRUE                 -- Only include events that are actual pitches
ORDER BY
    g.official_date ASC, p.inning ASC, p.pitch_index ASC;

-- Refresh MV
REFRESH MATERIALIZED VIEW mlb_tech_summit.mlb_gumbo_gold.game_report_mv;
